# CSC 575 Final Project: Interactive Local Text Document Search Engine

This notebook implements a local information retrieval system using the 20 Newsgroups dataset. The system indexes a local text collection and compares multiple retrieval methods: TF-IDF, semantic search, hybrid retrieval, and LLM-based query expansion.

In [121]:
import os
import re
import math
import pickle
import numpy as np
from collections import defaultdict, Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

for the dataset, I am using 20 newsgroups.

In [122]:
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes")
)

documents = dataset.data
labels = dataset.target
target_names = dataset.target_names

print("Total documents:", len(documents))
print("Total categories:", len(target_names))
print(target_names)

Total documents: 18846
Total categories: 20
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


## Dataset Subset

I use 5,000 documents from the 20 Newsgroups dataset. This satisfies the project requirement of indexing at least 2,000 files while keeping the system manageable in Google Colab.

In [123]:
NUM_DOCS = 5000

documents = documents[:NUM_DOCS]
labels = labels[:NUM_DOCS]

print("Using documents:", len(documents))

Using documents: 5000


note here, since i am using google colab, it removes the files once the runtime disconnects.

so if you are using jupyter notebook on your local system, i suggest that you check if there is an existing folder, before running the below code.

In [124]:
folder_path = "newsgroup_docs"
os.makedirs(folder_path, exist_ok=True)

for i, text in enumerate(documents):
    file_path = os.path.join(folder_path, f"doc_{i}.txt")
    with open(file_path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(text)

print("Saved", len(os.listdir(folder_path)), "files.")

Saved 5000 files.


## Text Preprocessing

The preprocessing step converts text to lowercase, tokenizes words, removes stopwords, and removes very short terms. The same preprocessing is applied to both documents and user queries.

In [88]:
stopwords = set("""
a an the and or but if while is are was were be been being to of in on for from with as by at this that these those
it its into about not no can could should would will just do does did have has had i you he she they we them his her their our
""".split())

def preprocess(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
    return tokens

## Inverted Index Construction

The inverted index stores each term as a key and maps it to the documents where it appears, along with term frequency information. This allows the system to retrieve candidate documents efficiently.

In [89]:
inverted_index = defaultdict(dict)
doc_lengths = {}
doc_term_freqs = {}

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        doc_id = int(filename.replace("doc_", "").replace(".txt", ""))
        file_path = os.path.join(folder_path, filename)

        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        tokens = preprocess(text)
        term_freq = Counter(tokens)

        doc_term_freqs[doc_id] = term_freq
        doc_lengths[doc_id] = len(tokens)

        for term, freq in term_freq.items():
            inverted_index[term][doc_id] = freq

print("Total indexed documents:", len(doc_term_freqs))
print("Vocabulary size:", len(inverted_index))

Total indexed documents: 5000
Vocabulary size: 46866


## IDF and TF-IDF Vector Creation

After building the inverted index, the system calculates inverse document frequency values and creates normalized TF-IDF vectors for each document.

In [90]:
N = len(doc_term_freqs)
idf = {}

for term, postings in inverted_index.items():
    df = len(postings)
    idf[term] = math.log10(N / df)

print("IDF computed for", len(idf), "terms.")

IDF computed for 46866 terms.


In [91]:
doc_vectors = {}

for doc_id, term_freq in doc_term_freqs.items():
    vector = {}

    for term, tf in term_freq.items():
        vector[term] = tf * idf[term]

    norm = math.sqrt(sum(weight ** 2 for weight in vector.values()))

    if norm != 0:
        for term in vector:
            vector[term] /= norm

    doc_vectors[doc_id] = vector

print("TF-IDF vectors created.")

TF-IDF vectors created.


In [92]:
def search(query, top_k=10):
    query_tokens = preprocess(query)
    query_tf = Counter(query_tokens)

    query_vector = {}
    for term, tf in query_tf.items():
        if term in idf:
            query_vector[term] = tf * idf[term]

    query_norm = math.sqrt(sum(weight ** 2 for weight in query_vector.values()))

    if query_norm != 0:
        for term in query_vector:
            query_vector[term] /= query_norm

    candidate_docs = set()
    for term in query_vector:
        if term in inverted_index:
            candidate_docs.update(inverted_index[term].keys())

    scores = {}

    for doc_id in candidate_docs:
        score = 0
        for term, q_weight in query_vector.items():
            score += q_weight * doc_vectors[doc_id].get(term, 0)
        scores[doc_id] = score

    ranked_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return ranked_results[:top_k]

below cell is only for testing

In [93]:
# query = "space nasa orbit shuttle"
# results = search(query, top_k=10)

# for rank, (doc_id, score) in enumerate(results, start=1):
#     print(rank, "Doc:", doc_id, "Score:", round(score, 4), "Category:", target_names[labels[doc_id]])
#     print(documents[doc_id][:300].replace("\n", " "))
#     print("-" * 80)

In [94]:
index_data = {
    "inverted_index": dict(inverted_index),
    "idf": idf,
    "doc_vectors": doc_vectors,
    "doc_lengths": doc_lengths,
    "labels": labels,
    "target_names": target_names
}

with open("search_engine_index.pkl", "wb") as f:
    pickle.dump(index_data, f)

print("Index saved successfully.")

Index saved successfully.


In [95]:
with open("search_engine_index.pkl", "rb") as f:
    loaded_index = pickle.load(f)

print("Loaded index keys:", loaded_index.keys())

Loaded index keys: dict_keys(['inverted_index', 'idf', 'doc_vectors', 'doc_lengths', 'labels', 'target_names'])


## Preset Evaluation Setup

The following test queries are used to evaluate retrieval quality. Each query is paired with a relevant 20 Newsgroups category, which is used as the ground-truth label for evaluation.

In [96]:
test_queries = [
    ("space nasa orbit shuttle", "sci.space"),
    ("baseball game team player", "rec.sport.baseball"),
    ("hockey puck nhl game", "rec.sport.hockey"),
    ("computer graphics image rendering", "comp.graphics"),
    ("windows driver file microsoft", "comp.os.ms-windows.misc"),
    ("car engine speed fuel", "rec.autos"),
    ("motorcycle bike ride helmet", "rec.motorcycles"),
    ("religion god christian church", "soc.religion.christian"),
    ("medical doctor disease treatment", "sci.med"),
    ("electronics circuit voltage power", "sci.electronics")
]

for q, cat in test_queries:
    print(q, "->", cat)

space nasa orbit shuttle -> sci.space
baseball game team player -> rec.sport.baseball
hockey puck nhl game -> rec.sport.hockey
computer graphics image rendering -> comp.graphics
windows driver file microsoft -> comp.os.ms-windows.misc
car engine speed fuel -> rec.autos
motorcycle bike ride helmet -> rec.motorcycles
religion god christian church -> soc.religion.christian
medical doctor disease treatment -> sci.med
electronics circuit voltage power -> sci.electronics


In [97]:
def get_category_id(category_name):
    return list(target_names).index(category_name)

for query, category in test_queries:
    print(category, "=", get_category_id(category))

sci.space = 14
rec.sport.baseball = 9
rec.sport.hockey = 10
comp.graphics = 1
comp.os.ms-windows.misc = 2
rec.autos = 7
rec.motorcycles = 8
soc.religion.christian = 15
sci.med = 13
sci.electronics = 12


In [98]:
def precision_recall_f1_at_k(query, relevant_category, k=10):
    results = search(query, top_k=k)
    relevant_cat_id = get_category_id(relevant_category)

    retrieved_docs = [doc_id for doc_id, score in results]
    relevant_retrieved = [doc_id for doc_id in retrieved_docs if labels[doc_id] == relevant_cat_id]

    total_relevant_docs = sum(1 for label in labels if label == relevant_cat_id)

    precision = len(relevant_retrieved) / k if k > 0 else 0
    recall = len(relevant_retrieved) / total_relevant_docs if total_relevant_docs > 0 else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1, len(relevant_retrieved), total_relevant_docs

In [99]:
def average_precision(query, relevant_category, k=10):
    results = search(query, top_k=k)
    relevant_cat_id = get_category_id(relevant_category)

    num_relevant_found = 0
    precision_sum = 0

    for rank, (doc_id, score) in enumerate(results, start=1):
        if labels[doc_id] == relevant_cat_id:
            num_relevant_found += 1
            precision_sum += num_relevant_found / rank

    if num_relevant_found == 0:
        return 0

    return precision_sum / num_relevant_found

okay so below results look good. basic model is working

In [100]:
evaluation_results = []

for query, category in test_queries:
    precision, recall, f1, relevant_found, total_relevant = precision_recall_f1_at_k(
        query, category, k=10
    )

    ap = average_precision(query, category, k=10)

    evaluation_results.append({
        "Query": query,
        "Relevant Category": category,
        "Relevant Found@10": relevant_found,
        "Total Relevant Docs": total_relevant,
        "Precision@10": precision,
        "Recall@10": recall,
        "F1@10": f1,
        "AP@10": ap
    })

evaluation_results

[{'Query': 'space nasa orbit shuttle',
  'Relevant Category': 'sci.space',
  'Relevant Found@10': 10,
  'Total Relevant Docs': 243,
  'Precision@10': 1.0,
  'Recall@10': 0.0411522633744856,
  'F1@10': 0.07905138339920949,
  'AP@10': 1.0},
 {'Query': 'baseball game team player',
  'Relevant Category': 'rec.sport.baseball',
  'Relevant Found@10': 6,
  'Total Relevant Docs': 257,
  'Precision@10': 0.6,
  'Recall@10': 0.023346303501945526,
  'F1@10': 0.0449438202247191,
  'AP@10': 0.7438492063492065},
 {'Query': 'hockey puck nhl game',
  'Relevant Category': 'rec.sport.hockey',
  'Relevant Found@10': 10,
  'Total Relevant Docs': 258,
  'Precision@10': 1.0,
  'Recall@10': 0.03875968992248062,
  'F1@10': 0.07462686567164178,
  'AP@10': 1.0},
 {'Query': 'computer graphics image rendering',
  'Relevant Category': 'comp.graphics',
  'Relevant Found@10': 8,
  'Total Relevant Docs': 259,
  'Precision@10': 0.8,
  'Recall@10': 0.03088803088803089,
  'F1@10': 0.05947955390334573,
  'AP@10': 0.875545

for better view, pandas is always a loyal friend

In [101]:
import pandas as pd

results_df = pd.DataFrame(evaluation_results)

results_df

,Query,Relevant Category,Relevant Found@10,Total Relevant Docs,Precision@10,Recall@10,F1@10,AP@10
0,space nasa orbit shuttle,sci.space,10,243,1.0,0.041152,0.079051,1.000000
1,baseball game team player,rec.sport.baseball,6,257,0.6,0.023346,0.044944,0.743849
2,hockey puck nhl game,rec.sport.hockey,10,258,1.0,0.038760,0.074627,1.000000
3,computer graphics image rendering,comp.graphics,8,259,0.8,0.030888,0.059480,0.875546
4,windows driver file microsoft,comp.os.ms-windows.misc,9,271,0.9,0.033210,0.064057,0.962654
5,car engine speed fuel,rec.autos,8,247,0.8,0.032389,0.062257,0.807341
6,motorcycle bike ride helmet,rec.motorcycles,10,276,1.0,0.036232,0.069930,1.000000
7,religion god christian church,soc.religion.christian,5,288,0.5,0.017361,0.033557,0.491111
8,medical doctor disease treatment,sci.med,10,262,1.0,0.038168,0.073529,1.000000
9,electronics circuit voltage power,sci.electronics,9,284,0.9,0.031690,0.061224,0.906041


In [102]:
mean_precision = results_df["Precision@10"].mean()
mean_recall = results_df["Recall@10"].mean()
mean_f1 = results_df["F1@10"].mean()
map_score = results_df["AP@10"].mean()

print("Average Precision@10:", round(mean_precision, 4))
print("Average Recall@10:", round(mean_recall, 4))
print("Average F1@10:", round(mean_f1, 4))
print("MAP@10:", round(map_score, 4))

Average Precision@10: 0.85
Average Recall@10: 0.0323
Average F1@10: 0.0623
MAP@10: 0.8787


## Interactive Gradio Interface

The final interface allows users to search the document collection, evaluate a query, and compare retrieval methods side by side.

I suggest whoever is seeing my project, it is better to use it on google collab because it has gradio pre installed but if you are using a local IDE then install gradio using pip

In [103]:
!pip install gradio

In [104]:
import gradio as gr
import pandas as pd

## Semantic Search

This section adds semantic search using sentence embeddings. Documents and queries are converted into dense vectors and compared using cosine similarity.

In [105]:
!pip install sentence-transformers

In [106]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [107]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [108]:
clean_documents = []

for doc in documents:
    cleaned = doc.replace("\n", " ").strip()
    if len(cleaned) == 0:
        cleaned = "empty document"
    clean_documents.append(cleaned)

doc_embeddings = embedding_model.encode(
    clean_documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Document embeddings created:", doc_embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Document embeddings created: (5000, 384)


In [109]:
def semantic_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for doc_id in top_indices:
        results.append((int(doc_id), float(similarities[doc_id])))

    return results

## Hybrid Search

Hybrid search combines TF-IDF scores and semantic similarity scores using an alpha value. Higher alpha gives more weight to TF-IDF, while lower alpha gives more weight to semantic search.

In [110]:
def hybrid_search(query, top_k=10, alpha=0.5):
    """
    alpha controls the balance:
    alpha = 1.0 means only TF-IDF
    alpha = 0.0 means only Semantic
    alpha = 0.5 means equal combination
    """

    tfidf_results = search(query, top_k=100)
    semantic_results = semantic_search(query, top_k=100)

    tfidf_scores = {doc_id: score for doc_id, score in tfidf_results}
    semantic_scores = {doc_id: score for doc_id, score in semantic_results}

    all_doc_ids = set(tfidf_scores.keys()).union(set(semantic_scores.keys()))

    if len(tfidf_scores) > 0:
        max_tfidf = max(tfidf_scores.values())
    else:
        max_tfidf = 1

    if len(semantic_scores) > 0:
        max_semantic = max(semantic_scores.values())
    else:
        max_semantic = 1

    hybrid_scores = {}

    for doc_id in all_doc_ids:
        normalized_tfidf = tfidf_scores.get(doc_id, 0) / max_tfidf if max_tfidf != 0 else 0
        normalized_semantic = semantic_scores.get(doc_id, 0) / max_semantic if max_semantic != 0 else 0

        final_score = alpha * normalized_tfidf + (1 - alpha) * normalized_semantic
        hybrid_scores[doc_id] = final_score

    ranked_results = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_results[:top_k]

below cell is for testing only

In [111]:

# query = "space nasa moon"

# print("TF-IDF Results")
# for doc_id, score in search(query, top_k=5):
#     print(doc_id, round(score, 4), target_names[labels[doc_id]])

# print("\nSemantic Results")
# for doc_id, score in semantic_search(query, top_k=5):
#     print(doc_id, round(score, 4), target_names[labels[doc_id]])

# print("\nHybrid Results")
# for doc_id, score in hybrid_search(query, top_k=5, alpha=0.5):
#     print(doc_id, round(score, 4), target_names[labels[doc_id]])

In [112]:
!pip install transformers accelerate sentencepiece

## LLM-Based Query Expansion

This section uses FLAN-T5 to generate controlled keyword expansions for a query. The expanded query is then passed into the TF-IDF retrieval system.

In [113]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print("Model loaded on:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded on: cuda


In [114]:
def generate_query_expansion(query):
    prompt = (
        "Generate 8 useful search keywords related to this query. "
        "Return only keywords, no full sentences. Query: " + query
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated_text

In [115]:
def llm_expanded_search(query, top_k=10):
    generated_text = generate_query_expansion(query)

    expanded_query = (
        query + " " +
        query + " " +
        query + " " +
        query + " " +
        query + " " +
        query + " " +
        query + " " +
        generated_text
    )

    results = search(expanded_query, top_k=top_k)

    return results, generated_text, expanded_query

below cell is for testing only

In [116]:
# query = "space nasa moon"

# results, generated_text, expanded_query = llm_expanded_search(query, top_k=10)

# print("Original Query:")
# print(query)

# print("\nGenerated Expansion:")
# print(generated_text)

# print("\nTop Results:")
# for rank, (doc_id, score) in enumerate(results, start=1):
#     print(rank, doc_id, round(score, 4), target_names[labels[doc_id]])

In [117]:
def gradio_method_search(query, method, top_k, alpha):
    if query.strip() == "":
        return pd.DataFrame(columns=[
            "Rank", "Doc ID", "Score", "Category", "Method", "Generated Expansion", "Snippet"
        ])

    top_k = int(top_k)
    alpha = float(alpha)
    generated_expansion = ""

    if method == "TF-IDF":
        results = search(query, top_k=top_k)

    elif method == "Semantic":
        results = semantic_search(query, top_k=top_k)

    elif method == "Hybrid":
        results = hybrid_search(query, top_k=top_k, alpha=alpha)

    elif method == "LLM Query Expansion":
        results, generated_expansion, expanded_query = llm_expanded_search(query, top_k=top_k)

    rows = []

    for rank, (doc_id, score) in enumerate(results, start=1):
        snippet = documents[doc_id][:300].replace("\n", " ")

        rows.append({
            "Rank": rank,
            "Doc ID": doc_id,
            "Score": round(score, 4),
            "Category": target_names[labels[doc_id]],
            "Method": method,
            "Generated Expansion": generated_expansion,
            "Snippet": snippet
        })

    return pd.DataFrame(rows)

In [118]:
def evaluate_method_query(query, relevant_category, method, top_k, alpha):
    if query.strip() == "":
        empty_results = pd.DataFrame(columns=[
            "Rank", "Doc ID", "Score", "Category", "Relevant?", "Method", "Generated Expansion", "Snippet"
        ])
        empty_metrics = pd.DataFrame(columns=["Metric", "Score"])
        return empty_results, empty_metrics

    top_k = int(top_k)
    alpha = float(alpha)
    generated_expansion = ""

    if method == "TF-IDF":
        results = search(query, top_k=top_k)

    elif method == "Semantic":
        results = semantic_search(query, top_k=top_k)

    elif method == "Hybrid":
        results = hybrid_search(query, top_k=top_k, alpha=alpha)

    elif method == "LLM Query Expansion":
        results, generated_expansion, expanded_query = llm_expanded_search(query, top_k=top_k)

    relevant_cat_id = get_category_id(relevant_category)

    rows = []
    relevant_found = 0
    precision_sum = 0

    for rank, (doc_id, score) in enumerate(results, start=1):
        is_relevant = labels[doc_id] == relevant_cat_id

        if is_relevant:
            relevant_found += 1
            precision_sum += relevant_found / rank

        snippet = documents[doc_id][:300].replace("\n", " ")

        rows.append({
            "Rank": rank,
            "Doc ID": doc_id,
            "Score": round(score, 4),
            "Category": target_names[labels[doc_id]],
            "Relevant?": "Yes" if is_relevant else "No",
            "Method": method,
            "Generated Expansion": generated_expansion,
            "Snippet": snippet
        })

    total_relevant_docs = sum(1 for label in labels if label == relevant_cat_id)

    precision = relevant_found / top_k if top_k > 0 else 0
    recall = relevant_found / total_relevant_docs if total_relevant_docs > 0 else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    if relevant_found == 0:
        ap = 0
    else:
        ap = precision_sum / relevant_found

    result_df = pd.DataFrame(rows)

    metrics_df = pd.DataFrame({
        "Metric": [
            "Precision@" + str(top_k),
            "Recall@" + str(top_k),
            "F1@" + str(top_k),
            "Average Precision@" + str(top_k),
            "Relevant Found",
            "Total Relevant Docs"
        ],
        "Score": [
            round(precision, 4),
            round(recall, 4),
            round(f1, 4),
            round(ap, 4),
            relevant_found,
            total_relevant_docs
        ]
    })

    return result_df, metrics_df

In [119]:
def compare_all_methods(query, relevant_category, top_k, alpha):
    if query.strip() == "":
        return pd.DataFrame(columns=[
            "Method", "Precision", "Recall", "F1", "Average Precision", "Relevant Found"
        ])

    methods = ["TF-IDF", "Semantic", "Hybrid", "LLM Query Expansion"]
    rows = []

    for method in methods:
        _, metrics_df = evaluate_method_query(
            query,
            relevant_category,
            method,
            top_k,
            alpha
        )

        metric_dict = dict(zip(metrics_df["Metric"], metrics_df["Score"]))

        rows.append({
            "Method": method,
            "Precision": metric_dict.get("Precision@" + str(int(top_k)), 0),
            "Recall": metric_dict.get("Recall@" + str(int(top_k)), 0),
            "F1": metric_dict.get("F1@" + str(int(top_k)), 0),
            "Average Precision": metric_dict.get("Average Precision@" + str(int(top_k)), 0),
            "Relevant Found": metric_dict.get("Relevant Found", 0)
        })

    return pd.DataFrame(rows)

In [120]:
with gr.Blocks(title="Interactive Local IR Search Engine") as demo:
    gr.Markdown("# Interactive Local Text Document Search Engine")
    gr.Markdown(
        "This system searches a local document corpus using TF-IDF, semantic embeddings, hybrid retrieval, "
        "and LLM-based query expansion. It also evaluates retrieval quality using category-based relevance judgments."
    )

    with gr.Tab("Search"):
        query_input = gr.Textbox(
            label="Enter Search Query",
            placeholder="Example: space nasa moon"
        )

        method_input = gr.Radio(
            choices=["TF-IDF", "Semantic", "Hybrid", "LLM Query Expansion"],
            value="TF-IDF",
            label="Search Method"
        )

        top_k_input = gr.Slider(
            minimum=5,
            maximum=20,
            value=10,
            step=1,
            label="Number of Results"
        )

        alpha_input = gr.Slider(
            minimum=0,
            maximum=1,
            value=0.5,
            step=0.1,
            label="Hybrid Weight Alpha: 1 = TF-IDF, 0 = Semantic"
        )

        search_button = gr.Button("Search")

        search_output = gr.Dataframe(
            label="Ranked Search Results",
            wrap=True
        )

        search_button.click(
            fn=gradio_method_search,
            inputs=[query_input, method_input, top_k_input, alpha_input],
            outputs=search_output
        )

    with gr.Tab("Evaluate Your Query"):
        eval_query_input = gr.Textbox(
            label="Enter Query to Evaluate",
            placeholder="Example: space nasa moon"
        )

        category_dropdown = gr.Dropdown(
            choices=list(target_names),
            value="sci.space",
            label="Select Relevant Category"
        )

        eval_method_input = gr.Radio(
            choices=["TF-IDF", "Semantic", "Hybrid", "LLM Query Expansion"],
            value="TF-IDF",
            label="Evaluation Method"
        )

        eval_top_k_input = gr.Slider(
            minimum=5,
            maximum=20,
            value=10,
            step=1,
            label="Evaluate Top K Results"
        )

        eval_alpha_input = gr.Slider(
            minimum=0,
            maximum=1,
            value=0.5,
            step=0.1,
            label="Hybrid Weight Alpha"
        )

        eval_button = gr.Button("Evaluate Query")

        eval_results_output = gr.Dataframe(
            label="Search Results with Relevance Labels",
            wrap=True
        )

        eval_metrics_output = gr.Dataframe(
            label="Evaluation Metrics",
            wrap=True
        )

        gr.Markdown(
            "Note: For evaluation, the selected category is treated as the ground-truth relevant class. "
            "Results from that category are marked as relevant."
        )

        eval_button.click(
            fn=evaluate_method_query,
            inputs=[
                eval_query_input,
                category_dropdown,
                eval_method_input,
                eval_top_k_input,
                eval_alpha_input
            ],
            outputs=[eval_results_output, eval_metrics_output]
        )

    with gr.Tab("Compare Methods"):
        compare_query_input = gr.Textbox(
            label="Enter Query",
            placeholder="Example: space nasa moon"
        )

        compare_category_dropdown = gr.Dropdown(
            choices=list(target_names),
            value="sci.space",
            label="Select Relevant Category"
        )

        compare_top_k_input = gr.Slider(
            minimum=5,
            maximum=20,
            value=10,
            step=1,
            label="Top K"
        )

        compare_alpha_input = gr.Slider(
            minimum=0,
            maximum=1,
            value=0.5,
            step=0.1,
            label="Hybrid Weight Alpha"
        )

        compare_button = gr.Button("Compare Methods")

        compare_output = gr.Dataframe(
            label="Method Comparison",
            wrap=True
        )

        compare_button.click(
            fn=compare_all_methods,
            inputs=[
                compare_query_input,
                compare_category_dropdown,
                compare_top_k_input,
                compare_alpha_input
            ],
            outputs=compare_output
        )



demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2f2c99086bcd7e9fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
